# Nguzo.ai - Colab Runner

This notebook is the **single entry point** for reproducing the research artifacts in this repository.  It:

1. Clones the repo and installs the pinned dependency set.
2. Runs the unit-test suite to verify the environment is sane.
3. Acquires the Swahili corpus and runs the tokenizer benchmark.
4. (Optional) Runs the full training-pipeline notebooks 01-05 in sequence.

Runtime: ~3-5 minutes for steps 1-3 on the free Colab tier.  Step 4 requires a GPU runtime and 1-24 hours depending on model size.

## 1. Setup

In [ ]:
import os, sys, subprocess
REPO_URL = "https://github.com/MuchiriTimothyGitau/nguzo.ai.git"
REPO_DIR = "/content/nguzo.ai"

if not os.path.isdir(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
os.chdir(REPO_DIR)
sys.path.insert(0, REPO_DIR)
print("Working dir:", os.getcwd())
print("Files:", sorted(os.listdir("."))[:10], "...")

In [ ]:
# Install the research-only pinned dependencies (no torch/whisper yet).
!pip install -q -r requirements-lock.txt

## 2. Run the test suite

In [ ]:
!python -m pytest tests/ -v --tb=short

All 35 tests should pass.  If anything fails, the rest of this notebook will probably fail too.

## 3. Run the tokenizer benchmark

In [ ]:
!python ml/data_acquisition.py --output data/swahili_corpus.txt --max-chars 500000
!python ml/benchmark_tokenizer.py --corpus data/swahili_corpus.txt --output-dir results --vocab-size 16000

In [ ]:
# Display the human-readable benchmark report
from pathlib import Path
print(Path("results/tokenizer_benchmark.md").read_text(encoding="utf-8"))

## 4. (Optional) Full training pipeline

Only run this on a GPU runtime (Runtime -> Change runtime type -> T4 GPU).  It executes the original 5-stage pipeline that trains a 5M-350M parameter language model.  Expect 6-48 hours depending on the model size you pick in notebook 05.

In [ ]:
# Only run if you have a GPU.  Comment out the next two lines if not.
# !pip install -q -r requirements.txt  # full pipeline (torch, whisper, librosa, ...)
# for nb in sorted(Path("notebooks").glob("0*.ipynb")):
#     print(f"\n=== Executing {nb.name} ===\n")
#     subprocess.run(["jupyter", "nbconvert", "--to", "notebook", "--execute",
#                     "--inplace", "--ExecutePreprocessor.timeout=-1", str(nb)], check=True)
print("Step 4 is commented out by default.  Uncomment the cell above to run the full pipeline on a GPU runtime.")

## 5. Inspect the results

In [ ]:
import json
from pathlib import Path

report = json.loads(Path("results/tokenizer_benchmark.json").read_text(encoding="utf-8"))
print("Configuration:")
print(json.dumps(report["config"], indent=2))
print("\nVanilla BPE:")
print(json.dumps({k: v for k, v in report["vanilla_bpe"].items()
                    if k != "atomic_chunk_coverage"}, indent=2))
print("\nLinguistically-informed BPE:")
print(json.dumps({k: v for k, v in report["linguistically_informed_bpe"].items()
                    if k != "atomic_chunk_coverage"}, indent=2))
print("\nImprovement (B over A):")
print(json.dumps(report["improvement_b_over_a"], indent=2))